In [1]:
import pandas as pd
import duckdb

# make sure to replace 'file.parquet' with your file path
df = pd.read_parquet('./data/idMinMaxDepth.parquet')
df = df.head(10)  # FOR TESTING

# use the values in the dataset_id to search via
# duckdb for the @id to generate the JSON-LD with.

# NOTE:  Duckdb should be able to read the data straight from minio
# ref: https://duckdb.org/docs/stable/extensions/aws

def search_duckdb(x):
    x = duckdb.sql(f"SELECT url FROM read_json('./jsonld/obis_source/*.jsonld') WHERE url like '%{x}%'").fetchall()
    xs = [str(item[0]) for item in x]
    return(xs)

df['docid'] = df['dataset_id'].apply(lambda x: search_duckdb(x))

dfe = df.explode('docid')


In [5]:
dfe.head(50)

,dataset_id,min_depth,max_depth,docid
0,057208a1-7eb2-4ea2-b3d9-988605e12229,49.0,248.0,https://obis.org/dataset/057208a1-7eb2-4ea2-b3...
1,14661428-e562-4259-8a1d-377544eeb75b,NaN,NaN,https://obis.org/dataset/14661428-e562-4259-8a...
1,14661428-e562-4259-8a1d-377544eeb75b,NaN,NaN,https://obis.org/dataset/14661428-e562-4259-8a...
2,2e653f52-25c7-4f72-9674-fa75ec5dca51,0.0,20.0,https://obis.org/dataset/2e653f52-25c7-4f72-96...
3,2efab3d2-83a8-49a4-bd2d-025434e9df90,NaN,10.0,https://obis.org/dataset/2efab3d2-83a8-49a4-bd...
3,2efab3d2-83a8-49a4-bd2d-025434e9df90,NaN,10.0,https://obis.org/dataset/2efab3d2-83a8-49a4-bd...
4,2fd06604-7d20-4712-9f68-eae9ae336bb5,NaN,NaN,https://obis.org/dataset/2fd06604-7d20-4712-9f...
4,2fd06604-7d20-4712-9f68-eae9ae336bb5,NaN,NaN,https://obis.org/dataset/2fd06604-7d20-4712-9f...
5,3f7a3e8d-f0ea-4786-90b8-da7194d87163,NaN,NaN,https://obis.org/dataset/3f7a3e8d-f0ea-4786-90...
5,3f7a3e8d-f0ea-4786-90b8-da7194d87163,NaN,NaN,https://obis.org/dataset/3f7a3e8d-f0ea-4786-90...


In [3]:
# remove where columns min/max have Nan
dfe_strict = dfe.dropna(subset=['max_depth', 'min_depth'], how='any')

In [4]:
dfe_strict.head(10)

,dataset_id,min_depth,max_depth,docid
0,057208a1-7eb2-4ea2-b3d9-988605e12229,49.0,248.0,https://obis.org/dataset/057208a1-7eb2-4ea2-b3...
2,2e653f52-25c7-4f72-9674-fa75ec5dca51,0.0,20.0,https://obis.org/dataset/2e653f52-25c7-4f72-96...
6,4c94cee2-f2f0-4a7b-825a-967bd991ca68,0.0,3000.0,https://obis.org/dataset/4c94cee2-f2f0-4a7b-82...
6,4c94cee2-f2f0-4a7b-825a-967bd991ca68,0.0,3000.0,https://obis.org/dataset/4c94cee2-f2f0-4a7b-82...


In [6]:
def populate_template(row):
    template = """ {{
      "@context": {{
        "@vocab": "https://schema.org/"
      }},
      "@id": "{docid}",
      "@type": "Dataset",
      "variableMeasured": [
        {{
          "@type": "PropertyValue",
          "name": "depth",
          "description": "Parsed and validated by OBIS.",
          "minValue": "{MIN}",
          "maxValue": "{MAX}",
          "propertyID": "https://obis.org/data/access/",
          "measurementTechnique": "Parsed and validated by OBIS.",
          "unitText": "m",
          "unitCode": [
            "https://qudt.org/vocab/unit/M", "https://vocab.nerc.ac.uk/collection/P06/current/ULAA/",
            "http://dbpedia.org/resource/Metre"
          ]
        }}
      ]
    }}
    """
    
    return template.format(MAX=row['max_depth'], MIN=row['min_depth'],  docid=row['docid'])




### DO NOT USE  All elements, where the min or max might be NONE

In [7]:
dfe = dfe.assign(jsonld=dfe.apply(populate_template, axis=1))


In [15]:

for index, row in dfe.iterrows():
    filename = str('./jsonld/output_all/' + row['dataset_id']) + '_depth.jsonld'  # adjust file extension as per your requirement
    with open(filename, 'w') as f:
        f.write(row['jsonld'])

### Only elements where the min and max have values


In [16]:
dfe_strict = dfe_strict.assign(jsonld=dfe_strict.apply(populate_template, axis=1))


In [17]:
for index, row in dfe_strict.iterrows():
    filename = str('./jsonld/output_strict/' + row['dataset_id']) + '_depth.jsonld'  # adjust file extension as per your requirement
    with open(filename, 'w') as f:
        f.write(row['jsonld'])